# Distance-based clustering of cohorts

Builds on `cohort_tests copy.ipynb` **Experiment 3** (cohort distance structure). That
test verified the *correlation* between true and fitted pairwise cohort distances — but
correlation does not say whether cohorts form **separable groups** or just smear along a
continuum. Here we put clustering on top of the distance matrix to ask:

1. **Cohort-level** — given the `(C, K)` cohort positions `E[gamma·delta]`, do standard
   clustering algorithms (agglomerative, k-means, spectral) recover meaningful groups?
   Quality is measured by **silhouette score** on the precomputed distance matrix.
2. **Sample-level** — given `Z_hat` for all `N` samples, do clusters in factor space
   align with the true cohort label? Quality is measured by **ARI**, **NMI**, and
   silhouette against true labels.
3. **Visualization** — 2D **MDS** (distance-preserving) on the cohort distance matrix
   and **PCA** on `Z_hat` to sanity-check that the clustering numbers reflect what is
   actually visible in the embedding.

The analysis runs twice: on synthetic Exp 3 data (where ground-truth `mu_ck` is known)
and on real COVID multi-omics data in `per_severity` mode (where WHO Ordinal Scale is
the soft ground-truth).

UMAP is intentionally skipped — `umap-learn` is not in `pyproject.toml`. MDS is the
natural distance-matrix embedding; PCA covers the sample scatter.


In [ ]:
import os, sys, warnings
sys.path.insert(0, os.path.abspath('.'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.optimize import linear_sum_assignment
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform

from sklearn.cluster import AgglomerativeClustering, KMeans, SpectralClustering
from sklearn.decomposition import PCA
from sklearn.manifold import MDS
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)

from src import FACTModel
from src.enums import Likelihood, WPrior, ZPrior
from src.model_config import CohortPriorConfig, ModelConfig, SimpleViewConfig
from src.views import Views

from covid_data.load_covid import load_covid


In [ ]:
def generate_data(mu_ck, sigma, n_per_cohort, dims=(15, 15), noise=1.0, seed=0):
    """Synthetic cohort data: Z ~ N(mu_ck[c], sigma), Y_m = Z W_m^T + noise."""
    rng = np.random.default_rng(seed)
    C, K = mu_ck.shape
    codes = np.repeat(np.arange(C), n_per_cohort)
    Z = np.array([rng.normal(mu_ck[c], sigma) for c in codes])
    Ys = [Z @ rng.normal(size=(d, K)).T + rng.normal(scale=noise, size=(Z.shape[0], d))
          for d in dims]
    return Views.from_list(Ys, cohorts=np.array([f'c{c}' for c in codes])), Z, codes


def fit_cohort(views, K, pi=0.3, max_iter=100, seed=0):
    cfg = ModelConfig(
        simple_view_configs=[SimpleViewConfig(likelihood=Likelihood.NORMAL,
                                              w_prior=WPrior.ARD_SS)
                              for _ in range(views.num_simple)],
        structured_view_configs=[],
        z_priors=[ZPrior.COHORT] * K,
        cohort_prior_config=CohortPriorConfig(pi=pi),
    )
    m = FACTModel(views=views, K=K, model_config=cfg, seed=seed)
    m.fit(max_iter=max_iter, pretrain=True, elbo_tres=0.0)
    return m


def cohort_dist(M):
    """Pairwise L2 distance between rows of M (cohort positions in latent space)."""
    diffs = M[:, None, :] - M[None, :, :]
    return np.sqrt(np.sum(diffs ** 2, axis=-1))


def cohort_positions(model, K):
    """(C, K) cohort positions in latent space from E[gamma * delta]."""
    pr = model.fa.node_z.z_priors
    return np.column_stack([pr[k].E_gamma * pr[k].E_delta for k in range(K)])


def cluster_cohorts(D, k_range=(2, 3, 4)):
    """Run agglomerative + k-means + spectral on a (C, C) distance matrix.

    Returns a DataFrame indexed by (method, n_clusters) with silhouette score and
    a dict of labels keyed the same way. Silhouette uses the precomputed distance
    matrix where the algorithm supports it; for k-means we reconstruct via the
    MDS-embedded coordinates so silhouette stays comparable.
    """
    C = D.shape[0]
    rows, labels = [], {}
    # Recover Euclidean coords for k-means via classical MDS — same metric as D
    coords = MDS(n_components=min(C - 1, 5), dissimilarity='precomputed',
                 random_state=0, n_init=4).fit_transform(D)
    for k in k_range:
        if k >= C:
            continue
        # Agglomerative on precomputed distance, average linkage
        agg = AgglomerativeClustering(n_clusters=k, metric='precomputed',
                                      linkage='average').fit(D)
        sil_agg = silhouette_score(D, agg.labels_, metric='precomputed') \
            if k > 1 and len(set(agg.labels_)) > 1 else np.nan
        rows.append({'method': 'agglomerative', 'k': k, 'silhouette': sil_agg})
        labels[('agglomerative', k)] = agg.labels_

        km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(coords)
        sil_km = silhouette_score(D, km.labels_, metric='precomputed') \
            if k > 1 and len(set(km.labels_)) > 1 else np.nan
        rows.append({'method': 'kmeans', 'k': k, 'silhouette': sil_km})
        labels[('kmeans', k)] = km.labels_

        # Spectral with affinity from distance (RBF kernel)
        gamma = 1.0 / (2 * (D[D > 0].mean() ** 2 + 1e-9))
        A = np.exp(-gamma * D ** 2)
        sp = SpectralClustering(n_clusters=k, affinity='precomputed',
                                random_state=0, assign_labels='kmeans').fit(A)
        sil_sp = silhouette_score(D, sp.labels_, metric='precomputed') \
            if k > 1 and len(set(sp.labels_)) > 1 else np.nan
        rows.append({'method': 'spectral', 'k': k, 'silhouette': sil_sp})
        labels[('spectral', k)] = sp.labels_

    return pd.DataFrame(rows), labels


def cluster_samples(Z, true_labels, k_range=(2, 3, 4)):
    """Cluster N samples in Z; report silhouette + ARI/NMI vs true_labels.

    Runs k-means and agglomerative (ward linkage on Euclidean Z).
    """
    rows, labels = [], {}
    true = np.asarray(true_labels)
    for k in k_range:
        km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z)
        agg = AgglomerativeClustering(n_clusters=k, linkage='ward').fit(Z)
        for name, lab in [('kmeans', km.labels_), ('agglomerative', agg.labels_)]:
            sil = silhouette_score(Z, lab) if len(set(lab)) > 1 else np.nan
            rows.append({'method': name, 'k': k, 'silhouette': sil,
                         'ARI_vs_true': adjusted_rand_score(true, lab),
                         'NMI_vs_true': normalized_mutual_info_score(true, lab)})
            labels[(name, k)] = lab
    # Also report silhouette of the TRUE labels themselves — upper bound on what
    # any clustering method on this Z can achieve while respecting the true grouping.
    if len(set(true)) > 1:
        sil_true = silhouette_score(Z, true)
        rows.append({'method': 'TRUE_LABELS', 'k': len(set(true)),
                     'silhouette': sil_true, 'ARI_vs_true': 1.0, 'NMI_vs_true': 1.0})
    return pd.DataFrame(rows), labels


---
## Section A — Synthetic (mirror of `cohort_tests copy.ipynb` Exp 3)

`C=5` cohorts in `K=3` latent dimensions, gradient mean structure on `Z_0`. We know
`mu_ck`, so `D_true = cohort_dist(mu_ck)` is the ground truth. We expect the cohort-level
silhouette to be high (the cohorts sit on a clean gradient) and the sample-level ARI
against true codes to be positive.


In [ ]:
C, K = 5, 3
mu_syn = np.array([
    [ 0.9,  0.0, 0.0],
    [ 1.0,  0.0, 0.0],
    [ 0.0,  0.0, 0.0],
    [-1.5,  0.0, 0.0],
    [-2.0,  0.0, 0.0],
])
views_syn, Z_syn, codes_syn = generate_data(
    mu_syn, sigma=1.2, n_per_cohort=[120] * C, seed=4
)
m_syn = fit_cohort(views_syn, K=K, pi=0.3, max_iter=100, seed=4)

M_fit_syn = cohort_positions(m_syn, K)
D_true_syn = cohort_dist(mu_syn)
D_fit_syn  = cohort_dist(M_fit_syn)

tri = np.triu_indices(C, 1)
dist_corr = float(np.corrcoef(D_true_syn[tri], D_fit_syn[tri])[0, 1])
print(f'Synthetic — Exp 3 distance correlation (upper triangle) = {dist_corr:.3f}')


In [ ]:
# Cohort-level clustering on the fitted distance matrix
df_syn_c, labels_syn_c = cluster_cohorts(D_fit_syn, k_range=(2, 3, 4))
print('Cohort-level clustering on D_fit (synthetic):')
print(df_syn_c.round(3).to_string(index=False))

# Dendrograms: fitted vs true distance matrix (average linkage)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, D, title in zip(axes, [D_fit_syn, D_true_syn], ['D_fit', 'D_true']):
    Z_link = linkage(squareform(D, checks=False), method='average')
    dendrogram(Z_link, ax=ax, labels=[f'c{c}' for c in range(C)],
               color_threshold=0.7 * Z_link[:, 2].max())
    ax.set_title(f'Dendrogram — {title} (average linkage)')
    ax.set_ylabel('linkage distance')
plt.tight_layout()
plt.show()


In [ ]:
# MDS embedding of fitted vs true distance matrices, with agglomerative k=3 labels
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
chosen_labels = labels_syn_c[('agglomerative', 3)]
for ax, D, title in zip(axes, [D_fit_syn, D_true_syn], ['D_fit', 'D_true']):
    coords = MDS(n_components=2, dissimilarity='precomputed',
                 random_state=0, n_init=4).fit_transform(D)
    palette = sns.color_palette('tab10', n_colors=len(set(chosen_labels)))
    sns.scatterplot(x=coords[:, 0], y=coords[:, 1], hue=chosen_labels,
                    palette=palette, s=120, ax=ax, legend='full')
    for i in range(C):
        ax.annotate(f'c{i}', (coords[i, 0], coords[i, 1]),
                    xytext=(6, 4), textcoords='offset points', fontsize=9)
    ax.set_title(f'MDS({title}) — agglomerative k=3')
    ax.set_xlabel('MDS-1')
    ax.set_ylabel('MDS-2')
plt.tight_layout()
plt.show()


In [ ]:
# Sample-level clustering on Z_hat
Z_hat_syn = m_syn.get_latent_factors()
df_syn_s, labels_syn_s = cluster_samples(Z_hat_syn, codes_syn, k_range=(2, 3, 5))
print('Sample-level clustering on Z_hat (synthetic):')
print(df_syn_s.round(3).to_string(index=False))

# PCA(2) on Z_hat: true codes vs best clustering (kmeans k=5 — matches C)
pca = PCA(n_components=2).fit(Z_hat_syn)
P = pca.transform(Z_hat_syn)
best_lab = labels_syn_s[('kmeans', 5)]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
sns.scatterplot(x=P[:, 0], y=P[:, 1], hue=[f'c{c}' for c in codes_syn],
                palette='tab10', s=18, linewidth=0, ax=axes[0], legend='full')
axes[0].set_title('PCA(Z_hat) — TRUE cohort')
sns.scatterplot(x=P[:, 0], y=P[:, 1], hue=best_lab,
                palette='tab10', s=18, linewidth=0, ax=axes[1], legend='full')
axes[1].set_title(f'PCA(Z_hat) — kmeans k=5')
for ax in axes:
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.0f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.0f}%)')
plt.tight_layout()
plt.show()


---
## Section B — COVID `per_severity`

Real multi-omics data. Cohort labels are `sev0` … `sev7` (WHO Ordinal Scale); the
numeric `severity` field is the soft ground truth. Some severity levels are very small
(`sev2` has only ~2 samples) — their cohort positions are unstable, so cluster
boundaries that lump them with neighbours are sensible rather than buggy.


In [ ]:
K_c = 5
data = load_covid(cohort_mode='per_severity', standardize=True)
print(f'N={data.views.N}, D_metab={data.views.simple[0].D}, '
      f'D_prot={data.views.simple[1].D}')
print('Cohort counts:')
print(pd.Series(data.cohorts).value_counts().sort_index().to_string())

m_cov = fit_cohort(data.views, K=K_c, pi=0.3, max_iter=100, seed=0)
M_fit_cov = cohort_positions(m_cov, K_c)

# Use cohort labels in the order the prior stores them (sorted unique).
cohort_names = np.array(sorted(np.unique(data.cohorts)))
sev_per_cohort = np.array([int(c[3:]) for c in cohort_names])

D_fit_cov = cohort_dist(M_fit_cov)
D_sev_cov = np.abs(sev_per_cohort[:, None] - sev_per_cohort[None, :]).astype(float)

C_cov = len(cohort_names)
tri_cov = np.triu_indices(C_cov, 1)
dist_corr_cov = float(np.corrcoef(D_fit_cov[tri_cov], D_sev_cov[tri_cov])[0, 1])
print(f'\nCOVID per_severity — distance corr with |severity gap| = {dist_corr_cov:.3f}')


In [ ]:
df_cov_c, labels_cov_c = cluster_cohorts(D_fit_cov, k_range=(2, 3, 4))
print('Cohort-level clustering on D_fit (COVID per_severity):')
print(df_cov_c.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
Z_link = linkage(squareform(D_fit_cov, checks=False), method='average')
dendrogram(Z_link, ax=ax, labels=list(cohort_names),
           color_threshold=0.7 * Z_link[:, 2].max())
ax.set_title('COVID per_severity — dendrogram (average linkage on D_fit)')
ax.set_ylabel('linkage distance')
plt.tight_layout()
plt.show()


In [ ]:
# 2D MDS, color = severity, size ~ sqrt(cohort size)
coords_cov = MDS(n_components=2, dissimilarity='precomputed',
                 random_state=0, n_init=4).fit_transform(D_fit_cov)
counts = pd.Series(data.cohorts).value_counts().reindex(cohort_names).values
sizes = 40 + 25 * np.sqrt(counts.astype(float))

fig, ax = plt.subplots(figsize=(6.5, 5))
sc = ax.scatter(coords_cov[:, 0], coords_cov[:, 1], c=sev_per_cohort,
                cmap='mako', s=sizes, edgecolor='k', linewidth=0.6)
for i, name in enumerate(cohort_names):
    ax.annotate(f'{name} (n={counts[i]})',
                (coords_cov[i, 0], coords_cov[i, 1]),
                xytext=(6, 4), textcoords='offset points', fontsize=9)
plt.colorbar(sc, ax=ax, label='WHO severity')
ax.set_title('COVID per_severity — MDS(D_fit), color = severity')
ax.set_xlabel('MDS-1')
ax.set_ylabel('MDS-2')
plt.tight_layout()
plt.show()


In [ ]:
Z_hat_cov = m_cov.get_latent_factors()
df_cov_s, labels_cov_s = cluster_samples(Z_hat_cov, data.severity, k_range=(2, 3, 4))
print('Sample-level clustering on Z_hat (COVID per_severity), label = severity:')
print(df_cov_s.round(3).to_string(index=False))

# PCA(2) on Z_hat — severity (continuous) vs kmeans k=3
pca = PCA(n_components=2).fit(Z_hat_cov)
P = pca.transform(Z_hat_cov)
best_lab_cov = labels_cov_s[('kmeans', 3)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
sc = axes[0].scatter(P[:, 0], P[:, 1], c=data.severity, cmap='mako',
                     s=18, linewidth=0)
plt.colorbar(sc, ax=axes[0], label='WHO severity')
axes[0].set_title('PCA(Z_hat) — color = TRUE severity')
sns.scatterplot(x=P[:, 0], y=P[:, 1], hue=best_lab_cov,
                palette='tab10', s=18, linewidth=0, ax=axes[1], legend='full')
axes[1].set_title('PCA(Z_hat) — kmeans k=3')
for ax in axes:
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.0f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.0f}%)')
plt.tight_layout()
plt.show()


---
## Verdict

Use these readouts to decide whether distance-based clustering is a useful description
of the cohorts:

- **Synthetic cohort-level silhouette** should be > 0.3 at `k=2` or `k=3` — the Exp 3
  gradient gives clean cluster boundaries, so any clustering method should agree.
- **Synthetic sample-level ARI vs true codes** should be > 0.3 at `k=5`. Lower values
  mean the factor noise (`sigma=1.2`) is large enough to blur individual samples; this
  is a property of the test setup, not the model.
- **COVID distance corr with |severity gap|** mirrors Exp 5 of `covid_cohort_tests.ipynb`
  — a positive correlation (≈ 0.6–0.8) means the model preserves the WHO ordinal
  structure at the cohort level.
- **COVID MDS plot** is the visual sanity check: severity should map onto a smooth
  gradient along one MDS axis. If the colors are scrambled, the cohort distances are
  not encoding severity and the rest of the analysis is suspect.
- **COVID sample-level ARI/NMI vs severity** > 0 (even small) confirms severity is at
  least partly recoverable from `Z_hat` alone — the model has found cohort-aware factors.

The methodology is validated when all five hold; otherwise the failing item points to
which step (fit, distance, or clustering) is the weak link.
